# Intent Manifest Inference & Accuracy Model — Notebook Comparison Suite

This notebook evaluates candidate ML/AI models for **Intent Manifest Inference** (Primary Track) and **Intent Divergence Detection** (Secondary Track).

### Models Evaluated:
1. **Baseline**: Frequency-Threshold Heuristic ($N=1$)
2. **Model 1**: Statistical Pattern & Naive Bayes Purpose Miner (`StatisticalPatternModel`)
3. **Model 2**: Hybrid LLM / Semantic Schema-Bounded Extractor (`LLMHybridManifestModel` — **Recommended**)

In [ ]:
import sys
import os
import json
import pandas as pd

# Ensure src is in python path
sys.path.insert(0, os.path.abspath('..'))

from src.models.baseline_frequency import FrequencyBaselineModel
from src.models.statistical_ml import StatisticalPatternModel
from src.models.llm_hybrid import LLMHybridManifestModel
from src.divergence.intent_divergence import IntentDivergenceEngine
from src.evaluation.harness import EvaluationHarness

## 1. Load Gold Execution Traces & Seeded Divergence Dataset

In [ ]:
with open('../data/eval_dataset/eval_traces_gold.json', 'r') as f:
    gold_traces = json.load(f)
with open('../data/eval_dataset/seeded_divergence_set.json', 'r') as f:
    seeded_cases = json.load(f)

print(f"Loaded {len(gold_traces)} gold activity traces.")
print(f"Loaded {len(seeded_cases)} seeded divergence test cases.")

## 2. Train Models & Execute Benchmark Suite on Held-Out Test Set

In [ ]:
train_traces = gold_traces[:20]
test_traces = gold_traces[20:]

baseline_model = FrequencyBaselineModel(threshold_n=1)
model_stat_ml = StatisticalPatternModel()
model_stat_ml.fit(train_traces)
model_llm_hybrid = LLMHybridManifestModel()

harness = EvaluationHarness()

models = [baseline_model, model_stat_ml, model_llm_hybrid]
results = {}
for m in models:
    results[m.name] = harness.evaluate_manifest_model(m, test_traces)

# Display DataFrame Comparison
df_comp = pd.DataFrame(results).T[
    ['over_permissioning_rate', 'scope_recall_macro', 'scope_precision_macro', 
     'constraint_exact_match', 'purpose_f1', 'ece_calibration_error', 'avg_latency_ms']
]
df_comp

## 3. Evaluate Secondary Track Intent Divergence Engine

In [ ]:
div_engine = IntentDivergenceEngine()
div_res = harness.evaluate_divergence_engine(div_engine, seeded_cases)
print(json.dumps(div_res, indent=2))